# 03. Jenkins: DevOps capabilities

## Что агент пока не умеет

После `02 Workspace` агент умеет читать репозиторий, но Jenkins для него всё ещё внешний мир.

Теперь добавляем DevOps capabilities как `tools`, а не как shell-команды.

> Tool — это контролируемый контракт между рассуждением модели и детерминированным Python-кодом.


## Как выглядит tool

Модель не видит весь исходный код connector. Она видит контракт:

```text
name
description
JSON schema аргументов
structured result
```

Разделение ответственности:

```text
Model:
- понимает намерение;
- выбирает tool;
- формирует JSON arguments;
- интерпретирует result.

Python connector:
- хранит auth boundary;
- делает HTTP request;
- нормализует Jenkins response;
- возвращает безопасный JSON.
```


## `_backend` в Jenkins stage

Jenkins tools и `_backend` отвечают за разные поверхности:

```text
_backend
→ workspace/files/shell boundary

JENKINS_TOOLS
→ Jenkins API boundary
```

Поэтому агент не должен делать так:

```text
execute("curl -u $JENKINS_USER:$JENKINS_TOKEN ...")
```

Shell не наследует `.env`, а Jenkins credentials намеренно спрятаны внутри connector.


In [ ]:
from pathlib import Path
import sys

for candidate in (Path.cwd(), Path.cwd() / 'workshop_notebooks' / 'openclaw_path'):
    if (candidate / 'workshop_utils.py').exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

from workshop_utils import REPO_ROOT, print_stage_context, register_graphs, write_text
from connectors.jenkins import JENKINS_TOOLS, copy_jenkins_job, get_jenkins_job_info, trigger_jenkins_job
from agents.openclaw_03_jenkins_tools import _backend

print_stage_context()
print("Workspace backend:", type(_backend()).__name__)
print("Jenkins tools:", [tool.name for tool in JENKINS_TOOLS])


In [ ]:
print("get_jenkins_job_info schema:")
print(get_jenkins_job_info.args_schema.model_json_schema())

print("trigger_jenkins_job schema:")
print(trigger_jenkins_job.args_schema.model_json_schema())

print("copy_jenkins_job schema:")
print(copy_jenkins_job.args_schema.model_json_schema())


In [ ]:
print(trigger_jenkins_job.invoke({
    "parameters": {"OPENCLAW_SMOKE": "true"},
    "dry_run": True,
}))

print(copy_jenkins_job.invoke({
    "source_job_url": "https://devops.brojs.ru/job/marat/job/test01/",
    "new_job_name": "test02",
    "dry_run": True,
}))


In [ ]:
ENTRYPOINT = '''\
from __future__ import annotations

import os
from pathlib import Path

from deepagents import create_deep_agent
from dotenv import load_dotenv

from connectors.jenkins import JENKINS_TOOLS


DEFAULT_MODEL = "openrouter:tencent/hy3:free"
REPO_ROOT = Path(__file__).resolve().parents[1]

load_dotenv(REPO_ROOT / ".env")


def _workspace_root() -> Path:
    return Path(os.getenv("OPENCLAW_WORKSPACE", str(REPO_ROOT))).expanduser().resolve()


def _backend():
    root = _workspace_root()
    if os.getenv("OPENCLAW_ENABLE_LOCAL_SHELL") == "1":
        from deepagents.backends import LocalShellBackend

        return LocalShellBackend(
            root_dir=root,
            virtual_mode=True,
            inherit_env=False,
            timeout=120,
            max_output_bytes=80_000,
        )

    from deepagents.backends import FilesystemBackend

    return FilesystemBackend(root_dir=root, virtual_mode=True)


JENKINS_PROMPT = """\
You are OpenClaw at stage 03 Jenkins.
Respond in the user's language; default to Russian.

This graph keeps the workspace backend from stage 02 and adds Jenkins tools.

Explain the boundary clearly:
- tools are deterministic Python contracts exposed to the model through name,
  description, and JSON schema;
- Jenkins credentials live inside the Python connector;
- _backend is still responsible for workspace access;
- do not use shell, curl, env, or printenv to access Jenkins secrets.

Separate read actions from write actions.
Read tools can inspect job metadata and config.
Write tools can trigger builds, copy jobs, and create jobs from config.xml.
If the user explicitly asks for a real Jenkins mutation, call the matching
Jenkins tool with dry_run=False.
Return normalized operational summaries, not raw dumps.
"""


agent = create_deep_agent(
    model=os.getenv("OPENCLAW_MODEL", DEFAULT_MODEL),
    tools=JENKINS_TOOLS,
    system_prompt=JENKINS_PROMPT,
    backend=_backend(),
)
'''

entrypoint = write_text('agents/openclaw_03_jenkins_tools.py', ENTRYPOINT)
config_path = register_graphs({
    'openclaw_03_jenkins': './agents/openclaw_03_jenkins_tools.py:agent'
})

print("Entrypoint:", entrypoint.relative_to(REPO_ROOT))
print("LangGraph config:", config_path.relative_to(REPO_ROOT))


## Проверка в LangGraph Studio

### Read-запрос

```text
Проверь Jenkins job и назови статус последней сборки. Используй Jenkins tools, не shell/curl/env.
```

### Write-запрос

```text
Теперь запусти smoke build с OPENCLAW_SMOKE=true. Это реальный запуск, используй dry_run=false.
```

### Job management-запрос

```text
Скопируй Jenkins job test01 в папке marat в новую job test02. Это реальное действие, используй dry_run=false.
```

### Ожидаемое поведение

1. Read вызывает `get_jenkins_job_info`.
2. Build вызывает `trigger_jenkins_job`.
3. Copy вызывает `copy_jenkins_job`.
4. Агент не пытается прочитать Jenkins secrets через shell.

### Текущее ограничение

DevOps capability есть, но пользователь всё ещё должен находиться в Studio. Следующий шаг — вынести интерфейс в VK.
